In [0]:
from pyspark.sql.functions import col, to_timestamp

print("--- Processando Tabela: ORDERS (Tratamento de Datas) ---")

# 1. Ler da Bronze (Agora lemos a TABELA, não o arquivo)
df_orders = spark.read.table("olist_portfolio.bronze.orders")

# 2. Transformação: Converter Strings para Timestamp
# O formato dos dados é "yyyy-MM-dd HH:mm:ss"
df_silver = df_orders.select(
    col("order_id"),
    col("customer_id"),
    col("order_status"),
    to_timestamp(col("order_purchase_timestamp")).alias("purchase_date"),
    to_timestamp(col("order_approved_at")).alias("approved_date"),
    to_timestamp(col("order_delivered_carrier_date")).alias("delivered_carrier_date"),
    to_timestamp(col("order_delivered_customer_date")).alias("delivered_customer_date"),
    to_timestamp(col("order_estimated_delivery_date")).alias("estimated_delivery_date")
).dropDuplicates(["order_id"]) # Remove duplicatas baseadas no ID do pedido

# 3. Salvar na Silver
df_silver.write \
         .format("delta") \
         .mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("olist_portfolio.silver.orders")

print("Sucesso! Tabela olist_portfolio.silver.orders criada.")

In [0]:
# Lista das tabelas simples (que não são a orders)
tables_to_process = [
    "customers", 
    "geolocation", 
    "order_items", 
    "order_payments", 
    "products", 
    "sellers", 
    "product_category_name_translation"
]

print("--- Processando Tabelas Simples (Deduplicação) ---\n")

for table in tables_to_process:
    try:
        print(f"Processando: {table}...")
        
        # 1. Ler Bronze
        df = spark.read.table(f"olist_portfolio.bronze.{table}")
        
        # 2. Limpar (Remove linhas 100% iguais)
        df_clean = df.dropDuplicates()
        
        # 3. Salvar Silver
        df_clean.write.format("delta").mode("overwrite").saveAsTable(f"olist_portfolio.silver.{table}")
        
        print(f"  -> OK: silver.{table}")
        
    except Exception as e:
        print(f"  -> Pulei {table}: {e} (Talvez ela não exista na bronze?)")

In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, when, trim

print("--- Criando Tabela Silver: Reviews com NLP ---")

# 1. Ler os reviews brutos
df_reviews = spark.read.table("olist_portfolio.bronze.order_reviews")

# 2. Limpeza: Remover quem não escreveu nada e limpar texto
df_clean = (df_reviews
    .filter(col("review_comment_message").isNotNull()) 
    .filter(col("review_comment_message") != "")       
    .withColumn("clean_text", lower(col("review_comment_message")))
    .withColumn("clean_text", regexp_replace("clean_text", "[\n\r]", " ")) # Tira quebras de linha
    .withColumn("clean_text", trim(col("clean_text")))
)

# 3. Classificação de Sentimento (Regra de Negócio)
# Nota 1 ou 2 = Negativo
# Nota 3 = Neutro
# Nota 4 ou 5 = Positivo
df_final = df_clean.withColumn("sentiment_label", 
    when(col("review_score") <= 2, "negativo")
    .when(col("review_score") == 3, "neutro")
    .otherwise("positivo")
)

# 4. Salvar na Silver
df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("olist_portfolio.silver.reviews_nlp")

print("Sucesso! Tabela olist_portfolio.silver.reviews_nlp criada.")